# Name That LLM — offline candidate inference

Attach the completed candidate checkpoint Dataset and Qwen3-14B-Base model. Enable suitable GPU resources and turn Internet OFF. This notebook only performs inference. Install required packages beforehand through Kaggle dependency management or an attached wheel dataset, never the PPU vendor PyTorch build. Run a full-scale synthetic workload check before submitting; a tiny interactive test file is not a runtime benchmark.


In [ ]:
from pathlib import Path
import json, os, sys, time
import pandas as pd
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
BASE_MODEL = Path("/kaggle/input/models/qwen-lm/qwen-3/Transformers/14b-base/1")
# Set this explicitly if more than one candidate checkpoint is attached.
CHECKPOINT = None
if CHECKPOINT is None:
    candidates = []
    for path in Path("/kaggle/input").rglob("meta.json"):
        try:
            meta = json.loads(path.read_text())
            if meta.get("request", {}).get("task") == "qwen3_two_stage" and meta.get("stage") == "official_refit_all" and not meta.get("acceptance_only") and (path.parent / "predict_offline.py").exists():
                candidates.append(path.parent)
        except (ValueError, OSError):
            pass
    if len(candidates) != 1:
        raise RuntimeError(f"Attach exactly one completed candidate, or set CHECKPOINT explicitly: {candidates}")
    CHECKPOINT = candidates[0]
CHECKPOINT = Path(CHECKPOINT)
assert BASE_MODEL.is_dir(), "Attach the Qwen3-14B-Base model and verify BASE_MODEL"
sys.path.insert(0, str(CHECKPOINT))
from predict_offline import predict
from validate_submission import validate_submission
meta = json.loads((CHECKPOINT / "meta.json").read_text())
test = pd.read_csv("/kaggle/input/competitions/name-that-llm/test.csv", dtype={"id": str})
started = time.monotonic()
predictions, diagnostic = predict(str(BASE_MODEL), str(CHECKPOINT), test, batch_size=2, device="auto")
submission = pd.DataFrame({"id": test["id"], "model": predictions})
validate_submission(submission, test["id"], meta["labels"])
submission.to_parquet("/kaggle/working/submission.parquet", index=False)
validate_submission(pd.read_parquet("/kaggle/working/submission.parquet"), test["id"], meta["labels"])
print({"rows": len(test), "seconds": time.monotonic()-started, "blind_groups": diagnostic["group_count"], "output": "/kaggle/working/submission.parquet"})
